# Seance 11 — Finitions + projet final

Voir user guide :
- Customizing Visualizations : https://altair-viz.github.io/user_guide/customization.html
- Saving Altair Charts : https://altair-viz.github.io/user_guide/saving_charts.html
- Data Transformations (a lire, pas indispensable ici) : https://altair-viz.github.io/user_guide/transform/index.html

## Objectifs
- Choisir des couleurs coherentes et lisibles
- Ajouter des annotations simples
- Composer une mini-infographie
- Sauvegarder en HTML et PNG

Rappel : on garde les transformations cote donnees avec **pandas** (comptages, moyennes, proportions).

## Criteres de reussite (rendu final)

- code reproductible
- charge les donnees
- identifie et prepare les variables
- figure avec : titre explicite, sous-titre (exemple de lecture), source, mise en forme
- export dans un format adapte (png/html)

In [ ]:
%pip install altair "vl-convert-python>=1.6.0"

In [ ]:
import os
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
os.makedirs('output', exist_ok=True)

data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df = pd.read_csv(data_url, compression="gzip", low_memory=False)

## 1) Couleurs coherentes (Harris vs Trump)

On fixe un code couleur stable pour toute l'infographie :
- Harris : bleu
- Trump : rouge
- Autres : gris

In [ ]:
vote_palette = ['Harris', 'Trump', 'Autres']
vote_colors = ['#2563EB', '#DC2626', '#9CA3AF']

## 2) Annotation simple : moyenne sur une distribution

Exemple : interet pour la politique (`V241201`, categories 1-5).

On construit un barplot de la distribution (pandas), puis on ajoute une ligne rouge sur la moyenne (pandas).

In [ ]:
d_int = df.loc[df['V241201'] > 0, ['V241201']].copy()
dist_int = d_int['V241201'].value_counts().sort_index().reset_index()
dist_int.columns = ['interet', 'n']

moy_int = d_int['V241201'].mean()
moy_int

In [ ]:
bars = alt.Chart(dist_int).mark_bar(color='#0EA5E9').encode(
    x=alt.X('interet', type='ordinal', title='Interet pour la politique (1-5)'),
    y=alt.Y('n', type='quantitative', title='Nombre de repondants')
)

rule = alt.Chart(pd.DataFrame({'x': [moy_int]})).mark_rule(color='#DC2626', strokeWidth=2).encode(
    x=alt.X('x', type='quantitative')
)

text = alt.Chart(pd.DataFrame({'x': [moy_int], 'label': [f'Moyenne: {moy_int:.2f}']})).mark_text(
    align='left', dx=5, dy=-5, color='#DC2626'
).encode(
    x=alt.X('x', type='quantitative'),
    text=alt.Text('label', type='nominal')
)

alt.layer(bars, rule, text).properties(
    title=alt.TitleParams(
        text="Interet pour la politique",
        subtitle=[
            "La moyenne se situe autour de 3, avec une distribution assez etalee.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=280
)

### Hack-Time 1 (10 min)

Faites la meme annotation (moyenne) sur une autre variable numerique :
- `V241156` (thermometre Harris)
- `V241157` (thermometre Trump)

Astuce : filtrez 0-100.

In [ ]:
# Hack-Time 1 : votre code ici

## 3) Mini-infographie (composition)

Exemple percutant en comportement politique :
- vote (Harris/Trump) selon ideologie
- evaluations affectives (thermometres) selon ideologie

On construit 2 graphiques et on les met cote a cote.

In [ ]:
# 3a) Vote selon ideologie (proportions)
d_vote = df.loc[df['V241177'].between(1, 7) & (df['V242096x'] > 0), ['V241177', 'V242096x']].copy()
d_vote['ideologie'] = d_vote['V241177'].astype(int)
d_vote['vote_label'] = d_vote['V242096x'].astype(int).replace({1: 'Harris', 2: 'Trump', 3: 'Autres', 4: 'Autres', 5: 'Autres', 6: 'Autres'})

tab = pd.crosstab(d_vote['ideologie'], d_vote['vote_label'], normalize='index').reset_index()
tab_long = tab.melt(id_vars='ideologie', var_name='vote_label', value_name='proportion')

chart_vote = alt.Chart(tab_long).mark_bar().encode(
    x=alt.X('ideologie', type='ordinal', title='Ideologie (1-7)'),
    y=alt.Y('proportion', type='quantitative', title='Proportion', axis=alt.Axis(format='%')),
    color=alt.Color('vote_label', type='nominal', title='Vote', scale=alt.Scale(domain=vote_palette, range=vote_colors))
).properties(
    title=alt.TitleParams(text="Vote presidentiel selon ideologie", subtitle="Les electeurs plus conservateurs votent davantage Trump", anchor='start'),
    width=330,
    height=280
)

# 3b) Thermometre Harris moyen selon ideologie
d_th = df.loc[df['V241177'].between(1, 7) & df['V241156'].between(0, 100), ['V241177', 'V241156']].copy()
means = d_th.groupby('V241177', as_index=False)['V241156'].mean().rename(columns={'V241177': 'ideologie', 'V241156': 'harris_moy'})

chart_th = alt.Chart(means).mark_line(point=True, size=3, color='#2563EB').encode(
    x=alt.X('ideologie', type='ordinal', title='Ideologie (1-7)'),
    y=alt.Y('harris_moy', type='quantitative', title='Thermometre Harris (moyenne)', scale=alt.Scale(domain=[0, 100]))
).properties(
    title=alt.TitleParams(text="Evaluation moyenne de Harris", subtitle="La moyenne baisse quand l'ideologie augmente", anchor='start'),
    width=330,
    height=280
)

infographie = alt.hconcat(chart_vote, chart_th).properties(
    title=alt.TitleParams(
        text="Polarisation ideologique et vote (ANES 2024)",
        subtitle=[
            "Deux lectures : le vote varie avec l'ideologie, et l'evaluation affective aussi.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    )
)

infographie

### Hack-Time 2 (15 min)

Modifiez l'infographie :
- remplacez `chart_th` par un graphique sur Trump (`V241157`)
- gardez les axes 0-100
- ajustez titres/sous-titres

In [ ]:
# Hack-Time 2 : votre code ici

## 4) Sauvegarder (HTML / PNG)

On sauvegarde la mini-infographie.

Remarque : le PNG necessite `vl-convert-python` (installe en debut de notebook).

In [ ]:
infographie.save('output/infographie.html')
infographie.save('output/infographie.png', scale_factor=2.0)

sorted(os.listdir('output'))

## Hack-Time 3 (20 min) — Votre prototype final

Construisez votre propre mini-infographie avec :
- 2 graphiques coherents
- un titre general + sous-titre general + source
- export HTML ou PNG

Idees de questions :
- education -> vote
- revenu -> intention de voter
- ideologie -> evaluations des candidats
- age -> interet politique

In [ ]:
# Hack-Time 3 : votre infographie ici

## Checklist
- [ ] Le notebook s'execute de bout en bout
- [ ] Les variables sont identifiees (codebook)
- [ ] Nettoyage des valeurs manquantes / invalides
- [ ] Titre explicite
- [ ] Sous-titre avec exemple de lecture
- [ ] Source
- [ ] Export HTML/PNG